In [11]:
import os, sys
import numpy as np

# If your notebook is in notebooks/, this makes src imports work.
sys.path.append(os.path.abspath(".."))

from src.utils.data_loader import load_data, one_hot_encode
from src.ann.neural_network import NeuralNetwork
from src.ann.optimizers import Optimizer
from sklearn.model_selection import train_test_split
from types import SimpleNamespace

In [1]:
import wandb
wandb.login()

wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from C:\Users\Ahmed Husain Zaidi\_netrc.
wandb: Currently logged in as: cs25m009 (cs25m009-indian-institute-of-technology-madras) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

In [2]:
wandb.init(
    project="Assignment_1",
    entity=None
)

# 2.1 Data Exploration and Class Distribution (3 Marks)
Log a W&B Table containing 5 sample images from each of the 10 classes in the dataset.
Identify any classes that look visually similar in their raw form. How might this visual
similarity impact your model?

In [5]:
wandb.init(project="Assignment_1", name="part2_1_data_exploration")

# Load MNIST dataset
(X_train, y_train), _ = load_data("mnist")

# Create W&B table
table = wandb.Table(columns=["dataset", "class", "image"])

# Log 5 images per class
for class_id in range(10):
    indices = np.where(y_train == class_id)[0][:5]
    
    for idx in indices:
        image = X_train[idx].reshape(28, 28)
        table.add_data(
            "MNIST",
            class_id,
            wandb.Image(image, caption=f"class {class_id}")
        )

wandb.log({"MNIST_sample_images": table})
wandb.finish()

# 2.3 The Optimizer Showdown
Compare the convergence rates of all 4 optimizers using the same architecture (3 hidden
layers, 128 neurons each, ReLU activation). Which optimizer minimized the loss fastest in
the first 5 epochs? Theoretically, why does RMSProp often outperform standard SGD on
image classification?

In [7]:
(X_train, y_train), _ = load_data("mnist")
y_train = one_hot_encode(y_train)

# Fixed architecture required by assignment
hidden_layers = [128, 128, 128]

optimizers = ["sgd", "momentum", "nag", "rmsprop"]

for opt in optimizers:

    run = wandb.init(
        project="Assignment_1",
        name=f"optimizer_{opt}",
        config={
            "optimizer": opt,
            "hidden_layers": hidden_layers,
            "activation": "relu",
            "learning_rate": 0.01,
            "epochs": 5,
            "batch_size": 64
        }
    )

    # Build model
    model = NeuralNetwork(run.config)
    optimizer = Optimizer(opt, learning_rate=0.01)

    batch_size = 64
    epochs = 5
    num_samples = X_train.shape[0]

    for epoch in range(epochs):

        perm = np.random.permutation(num_samples)
        X_train = X_train[perm]
        y_train = y_train[perm]

        epoch_loss = 0

        for i in range(0, num_samples, batch_size):

            xb = X_train[i:i+batch_size]
            yb = y_train[i:i+batch_size]

            logits = model.forward(xb)
            loss = model.loss_fn.compute(yb, logits)

            model.backward(yb, logits)
            optimizer.step(model.layers)

            epoch_loss += loss

        epoch_loss /= (num_samples // batch_size)

        wandb.log({
            "epoch": epoch,
            "train_loss": epoch_loss
        })

    wandb.finish()

epoch,▁▃▅▆█
train_loss,█▃▂▁▁
epoch,4
train_loss,0.3611


epoch,▁▃▅▆█
train_loss,█▃▂▁▁
epoch,4
train_loss,0.12846


epoch,▁▃▅▆█
train_loss,█▃▂▁▁
epoch,4
train_loss,0.12424


epoch,▁▃▅▆█
train_loss,█▃▂▁▁
epoch,4
train_loss,0.11617


# 2.4 Vanishing Gradient Analysis
Fix the optimizer to RMSProp and compare Sigmoid and ReLU for different network
configurations. Log the gradient norms for the first hidden layer. Do you observe the
vanishing gradient problem with Sigmoid? Provide a plot to support your observation.

In [13]:
(X_full, y_full), _ = load_data("mnist")
X_train, X_val, y_train, y_val = train_test_split(
    X_full, y_full, test_size=0.1, random_state=42, stratify=y_full
)
y_train_oh = one_hot_encode(y_train)
y_val_oh = one_hot_encode(y_val)

# ---- experiment settings ----
ACTS = ["sigmoid", "relu"]
# "different network configurations" (depth changes, 128 neurons each)
CONFIGS = {
    "2x128": [128, 128],
    "3x128": [128, 128, 128],
    "5x128": [128, 128, 128, 128, 128],
}
EPOCHS = 5
BATCH = 64
LR = 1e-3
OPT = "rmsprop"  # fixed as per PDF

for cfg_name, hidden_sizes in CONFIGS.items():
    for act in ACTS:
        run = wandb.init(
            project="Assignment_1",
            name=f"2.4_{cfg_name}_{act}_{OPT}",
            tags=["part2", "2.4", "mnist"],
            config={
                "dataset": "mnist",
                "optimizer": OPT,
                "learning_rate": LR,
                "weight_decay": 0.0,
                "loss": "cross_entropy",
                "activation": act,
                "weight_init": "xavier",
                "num_hidden_layers": len(hidden_sizes),
                "hidden_sizes": hidden_sizes,
                "epochs": EPOCHS,
                "batch_size": BATCH,
                "configuration": cfg_name,
            }
        )

        args = SimpleNamespace(**dict(run.config))
        model = NeuralNetwork(args)
        model.optimizer = Optimizer(OPT, learning_rate=LR, weight_decay=0.0)

        n = X_train.shape[0]

        for epoch in range(EPOCHS):
            perm = np.random.permutation(n)
            X_shuf = X_train[perm]
            y_shuf = y_train_oh[perm]

            # track average gradient norm of FIRST hidden layer weights across batches
            grad_norm_sum = 0.0
            batches = 0

            # also log train loss (optional but useful)
            loss_sum = 0.0
            seen = 0

            for start in range(0, n, BATCH):
                xb = X_shuf[start:start + BATCH]
                yb = y_shuf[start:start + BATCH]

                logits = model.forward(xb)
                loss = model.loss_fn.compute(yb, logits)

                model.backward(yb, logits)

                # first hidden layer corresponds to layers[0]
                grad_norm_sum += np.linalg.norm(model.layers[0].grad_W)
                batches += 1

                model.update_weights()

                loss_sum += loss * xb.shape[0]
                seen += xb.shape[0]

            wandb.log({
                "epoch": epoch + 1,
                "train_loss": loss_sum / max(seen, 1),
                "grad_norm_first_hidden_layer": grad_norm_sum / max(batches, 1),
            })

        wandb.finish()

epoch,▁▃▅▆█
grad_norm_first_hidden_layer,▁█▇▇▆
train_loss,█▃▂▁▁
epoch,5
grad_norm_first_hidden_layer,0.17993
train_loss,0.10357


epoch,▁▃▅▆█
grad_norm_first_hidden_layer,█▄▃▂▁
train_loss,█▃▂▁▁
epoch,5
grad_norm_first_hidden_layer,0.55233
train_loss,0.04982


epoch,▁▃▅▆█
grad_norm_first_hidden_layer,▁▇███
train_loss,█▃▂▁▁
epoch,5
grad_norm_first_hidden_layer,0.23048
train_loss,0.10735


epoch,▁▃▅▆█
grad_norm_first_hidden_layer,█▄▂▁▁
train_loss,█▃▂▁▁
epoch,5
grad_norm_first_hidden_layer,0.7147
train_loss,0.05129


epoch,▁▃▅▆█
grad_norm_first_hidden_layer,▁█▇▇▆
train_loss,█▃▂▁▁
epoch,5
grad_norm_first_hidden_layer,0.3512
train_loss,0.16058


epoch,▁▃▅▆█
grad_norm_first_hidden_layer,█▃▂▂▁
train_loss,█▃▂▁▁
epoch,5
grad_norm_first_hidden_layer,0.9392
train_loss,0.06145


# 2.5 The ”Dead Neuron” Investigation
Using ReLU activation and a high learning rate (e.g., 0.1), monitor the activations of
your hidden layers. Find a run where the validation accuracy plateaus early. Look at the
distribution of your activations. Can you identify ”dead neurons” (neurons that output
zero for all inputs)? Compare this run with a Tanh run and explain the difference in
convergence based on the gradients you observed.

In [15]:
def accuracy_from_logits(logits, y_onehot):
    return float(np.mean(np.argmax(logits, axis=1) == np.argmax(y_onehot, axis=1)))

def forward_collect_hidden_activations(model, X):
    """
    Runs a forward pass and returns:
      - logits
      - list of hidden activations (post-activation), one per hidden layer
    Uses the model's own layers + activation modules (no softmax).
    """
    a = X
    hidden_acts = []
    for i, layer in enumerate(model.layers):
        z = layer.forward(a)
        if i < len(model.activations):          # hidden layers
            a = model.activations[i].forward(z) # post-activation
            hidden_acts.append(a)
        else:
            a = z                               # logits
    return a, hidden_acts

def dead_fraction_from_acts(A):
    """
    A shape: (N, H). A neuron is "dead" if it outputs 0 for ALL inputs in this batch.
    Returns fraction of dead neurons in that layer.
    """
    dead = np.all(A == 0, axis=0)  # per neuron
    return float(np.mean(dead))

# ---- Data ----
(X_full, y_full), _ = load_data("mnist")
X_tr, X_val, y_tr, y_val = train_test_split(X_full, y_full, test_size=0.1, random_state=42, stratify=y_full)
y_tr_oh, y_val_oh = one_hot_encode(y_tr), one_hot_encode(y_val)

# ---- Fixed architecture for 2.5 ----
hidden_sizes = [128, 128, 128]
epochs = 8
batch_size = 64
optimizer_name = "sgd"
lr_high = 0.1

for activation in ["relu", "tanh"]:
    run = wandb.init(
        project="Assignment_1",
        group="2.5_dead_neurons",
        name=f"2.5_{activation}_lr{lr_high}",
        tags=["part2", "2.5", "mnist"],
        config={
            "dataset": "mnist",
            "activation": activation,
            "optimizer": optimizer_name,
            "learning_rate": lr_high,
            "epochs": epochs,
            "batch_size": batch_size,
            "hidden_sizes": hidden_sizes,
            "weight_init": "xavier",
            "loss": "cross_entropy",
        },
    )

    # Build args object that NeuralNetwork expects (attribute access)
    class Args: pass
    args = Args()
    for k, v in dict(run.config).items():
        setattr(args, k, v)
    setattr(args, "num_hidden_layers", len(hidden_sizes))

    model = NeuralNetwork(args)
    model.optimizer = Optimizer(optimizer_name, learning_rate=lr_high, weight_decay=0.0)

    n = X_tr.shape[0]

    for ep in range(1, epochs + 1):
        perm = np.random.permutation(n)
        X_shuf = X_tr[perm]
        y_shuf = y_tr_oh[perm]

        loss_sum, seen = 0.0, 0

        # ---- Train for one epoch ----
        for s in range(0, n, batch_size):
            xb = X_shuf[s:s + batch_size]
            yb = y_shuf[s:s + batch_size]

            logits = model.forward(xb)
            loss = model.loss_fn.compute(yb, logits)
            loss_sum += loss * xb.shape[0]
            seen += xb.shape[0]

            model.backward(yb, logits)
            model.update_weights()

        train_loss = loss_sum / max(seen, 1)

        # ---- Validation metrics + analysis batch ----
        # Use a fixed subset for analysis to keep logging fast & consistent
        Xv = X_val[:2048]
        yv = y_val_oh[:2048]

        val_logits, hidden_acts = forward_collect_hidden_activations(model, Xv)
        val_acc = accuracy_from_logits(val_logits, yv)

        # Dead neuron fraction + activation distributions (histograms)
        log_dict = {
            "epoch": ep,
            "train_loss": train_loss,
            "val_accuracy": val_acc,
        }

        for li, A in enumerate(hidden_acts, start=1):
            # dead neurons: all zero across inputs
            if activation == "relu":
                log_dict[f"layer{li}_dead_fraction"] = dead_fraction_from_acts(A)

            # activation distribution
            log_dict[f"layer{li}_activation_hist"] = wandb.Histogram(A.flatten())

        # Gradient norms (use last backprop of epoch as proxy; good enough for report narrative)
        for li, layer in enumerate(model.layers[:-1], start=1):
            log_dict[f"layer{li}_gradW_norm"] = float(np.linalg.norm(layer.grad_W))

        wandb.log(log_dict)

    wandb.finish()

epoch,▁▂▃▄▅▆▇█
layer1_dead_fraction,▁▁▁▁▁▁▁▁
layer1_gradW_norm,▅▅▃▄▆▁▄█
layer2_dead_fraction,▁▁▁▁▁▁▁▁
layer2_gradW_norm,▅▅▄▅▆▁▅█
layer3_dead_fraction,██▁▁▁▁▁▁
layer3_gradW_norm,▇▆▄▆█▁▆█
train_loss,█▃▂▂▂▁▁▁
val_accuracy,▁▄▆▅▇▇▇█
epoch,8
layer1_dead_fraction,0


epoch,▁▂▃▄▅▆▇█
layer1_gradW_norm,▆▄█▃▇▅▁▅
layer2_gradW_norm,▅▃█▃█▅▁▄
layer3_gradW_norm,▅▃█▃▇▄▁▄
train_loss,█▄▃▂▂▁▁▁
val_accuracy,▁▄▄▇█▇██
epoch,8
layer1_gradW_norm,0.69648
layer2_gradW_norm,0.39133
layer3_gradW_norm,0.25826
train_loss,0.04772
